In [1]:
!pip install librosa


In [2]:
import numpy as np

fps = 3
duration_seconds = 30 
total_frames = fps * duration_seconds
cv_features = 512

dummy_cv_array = np.random.rand(total_frames, cv_features).astype(np.float32)

file_name = 'dummy_cv_output.npy'
np.save(file_name, dummy_cv_array)

print("CV OUTPUT SIMULATION")
print(f"File created: {file_name}")
print(f"Exact Shape of the data: {dummy_cv_array.shape}")
print(f"First 3 numbers of the very first frame: {dummy_cv_array[0, :3]}")

CV OUTPUT SIMULATION
File created: dummy_cv_output.npy
Exact Shape of the data: (90, 512)
First 3 numbers of the very first frame: [0.01621061 0.9142421  0.41544273]


In [3]:
import numpy as np
import librosa
duration_seconds = 30
fps = 3
target_frames = duration_seconds * fps  
n_mfcc = 128                            

sr = 22050  
hop_length = int(sr / fps)

raw_audio = np.random.randn(sr * duration_seconds).astype(np.float32)

audio_features = librosa.feature.mfcc(
    y=raw_audio, 
    sr=sr, 
    n_mfcc=n_mfcc, 
    hop_length=hop_length
).T

audio_features = audio_features[:target_frames, :]

file_name = 'dummy_audio_output.npy'
np.save(file_name, audio_features)

print("--- AUDIO EXTRACTION SIMULATION ---")
print(f"File created: {file_name}")
print(f"Exact Shape of the data: {audio_features.shape}")
print(f"First 3 MFCCs of the very first frame: {audio_features[0, :3]}")

--- AUDIO EXTRACTION SIMULATION ---
File created: dummy_audio_output.npy
Exact Shape of the data: (90, 128)
First 3 MFCCs of the very first frame: [157.6328    -8.558817  -3.767908]


In [4]:
import torch
import torch.nn as nn

class MultimodalProjector(nn.Module):
    def __init__(self):
        super(MultimodalProjector, self).__init__()
        
        # 1. The Video Compressor (512 -> 256)
        self.visual_translator = nn.Linear(in_features=512, out_features=256)
        
        # 2. The Audio Expander (128 -> 256)
        self.audio_translator = nn.Linear(in_features=128, out_features=256)
        
        # 3. Activation Function (The Filter)
        self.relu = nn.ReLU()

    def forward(self, cv_tensor, audio_tensor):
        # Step A: Push data through the translators
        projected_cv = self.visual_translator(cv_tensor)
        projected_audio = self.audio_translator(audio_tensor)
        
        # Step B: Apply the filter to remove negative noise
        clean_cv = self.relu(projected_cv)
        clean_audio = self.relu(projected_audio)
        
        return clean_cv, clean_audio

# --- HOW TO USE IT ---
# 1. Boot up the projector and send it to the GPU
projector = MultimodalProjector().cuda()

# 2. Convert your earlier NumPy arrays to PyTorch Tensors and send to GPU
# (Assuming dummy_cv_array and audio_features from earlier exist)
cv_gpu_tensor = torch.tensor(dummy_cv_array).cuda()
audio_gpu_tensor = torch.tensor(audio_features).cuda()

# 3. Do the actual translation!
final_cv, final_audio = projector(cv_gpu_tensor, audio_gpu_tensor)

print(f"Final Visual Shape: {final_cv.shape}")     
print(f"Final Audio Shape: {final_audio.shape}")   

Final Visual Shape: torch.Size([90, 256])
Final Audio Shape: torch.Size([90, 256])
